# 05 遺伝子のまとまりで見る（GSEA）

04 までは、遺伝子を1つずつ見てきました。ここでは、働きの近い遺伝子をまとめた**遺伝子セット**ごとに、そのセットの遺伝子が全体として shPPP3CA で増える側と減る側のどちらに偏っているかを調べます。この方法を GSEA（Gene Set Enrichment Analysis）と呼びます。

1つ1つの遺伝子の変化は小さくても、同じ働きの遺伝子がそろって同じ向きに動いていれば、GSEA で見つけられます。

## 準備

下のセルを実行して、配布元から `h.all.v2024.1.Hs.symbols.gmt` をダウンロードします。

In [ ]:
%pip install -q gseapy==1.3.1
import warnings
warnings.filterwarnings("ignore")  # 計算には影響しない警告を表示しない

DATA_URL = "https://pub-fcc6b90fb32448a4b3b455e664df76f0.r2.dev/e2f1-rnaseq/v1/"
!wget -q -nc {DATA_URL}h.all.v2024.1.Hs.symbols.gmt
!ls -l h.all.v2024.1.Hs.symbols.gmt

## 前のノートブックの結果を読み込む

04_Visualization の最後でダウンロードした `results_04.zip` を使います。下のセルを実行し、「ファイルを選択」を押して `results_04.zip` を選んでください。zip の中の `deseq2_results.csv` を、このノートブックで使います。

In [ ]:
import zipfile
import os
from pathlib import Path

PREV_ZIP = "results_04.zip"
NEED = ['deseq2_results.csv']

if not all(Path(n).exists() for n in NEED):
    if Path(PREV_ZIP).exists():
        zipfile.ZipFile(PREV_ZIP).extractall()
    else:
        from google.colab import files
        for name in files.upload():
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall()

missing = [n for n in NEED if not Path(n).exists()]
if missing:
    print("見つからないファイルがあります:", missing)
    print("このセルをもう一度実行して zip を選び直すか、下の「zip が手元にないとき」のセルを実行してください。")
else:
    print("読み込みました:", NEED)

### zip が手元にないとき

前のノートブックを最後まで実行できなかった人や、zip をなくした人だけ、下のセルを実行してください。教員が用意した `results_04.zip` をダウンロードして使います。

In [ ]:
!wget -q -nc {DATA_URL}results_04.zip
zipfile.ZipFile(PREV_ZIP).extractall()
print("読み込みました:", [n for n in NEED if Path(n).exists()])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

res = pd.read_csv("deseq2_results.csv", index_col="gene")
GMT = "h.all.v2024.1.Hs.symbols.gmt"

## 遺伝子セット

MSigDB という遺伝子セットのデータベースから、Hallmark と呼ばれる50個の遺伝子セットを使います。細胞周期、炎症、代謝など、よく知られた生物学的な過程ごとに遺伝子をまとめたものです。

In [ ]:
gene_sets = gp.read_gmt(GMT)
print("遺伝子セットの数:", len(gene_sets))
for name in list(gene_sets)[:5]:
    print(name, len(gene_sets[name]), "遺伝子")

In [ ]:
gene_sets["HALLMARK_E2F_TARGETS"][:20]

`HALLMARK_E2F_TARGETS` は、転写因子 E2F が働きを強める遺伝子の集まりです。

## 遺伝子に順位を付ける

すべての遺伝子を、DESeq2 の `stat`（log₂FC ÷ その不確かさ）の大きい順に並べます。上位ほど shPPP3CA で確かに増えた遺伝子、下位ほど確かに減った遺伝子です。

In [ ]:
ranking = res["stat"].dropna().sort_values(ascending=False)
print("順位を付けた遺伝子:", len(ranking))
print("上位5つ:")
print(ranking.head())
print("下位5つ:")
print(ranking.tail())

## GSEA を実行する

遺伝子セットごとに、そのセットの遺伝子が順位の上の方か下の方に偏っているかを調べます。偏りが偶然でも起きる程度かどうかは、遺伝子の並びをランダムに入れ替える計算（`permutation_num` 回）と比べて判断します。実行には1分ほどかかります。

In [ ]:
gsea = gp.prerank(
    rnk=ranking,
    gene_sets=GMT,
    permutation_num=1000,
    seed=0,
    threads=2,
    outdir=None,
    verbose=False,
)
table = gsea.res2d[["Term", "NES", "NOM p-val", "FDR q-val", "Lead_genes"]].copy()
table["NES"] = table["NES"].astype(float)
table["FDR q-val"] = table["FDR q-val"].astype(float)
table = table.sort_values("NES")
table[["Term", "NES", "FDR q-val"]].head(10)

結果の主な列の意味は次のとおりです。

| 列 | 意味 |
|---|---|
| `NES` | 偏りの強さ。負なら shPPP3CA で減る側、正なら増える側に偏っている |
| `FDR q-val` | たくさんの遺伝子セットを調べたことを考えて調整した確からしさ。小さいほど確か。0 と表示されるのは、1000回の入れ替えで一度もこれ以上の偏りが出なかったという意味で、本当に0というわけではありません |
| `Lead_genes` | 偏りを生んでいる中心の遺伝子（leading edge） |

上の表は NES の小さい順、つまり shPPP3CA で減る側に強く偏った遺伝子セットから並べています。増える側に偏った遺伝子セットは、次のセルで見られます。

In [ ]:
table[["Term", "NES", "FDR q-val"]].tail(10).iloc[::-1]

## NES を棒グラフにする

FDR q-val < 0.05 の遺伝子セットだけを並べます。

In [ ]:
FDR = 0.05

sig = table[table["FDR q-val"] < FDR]
fig, ax = plt.subplots(figsize=(7, 0.28 * len(sig) + 1))
ax.barh(sig["Term"].str.replace("HALLMARK_", ""), sig["NES"],
        color=["#2F6FC4" if v < 0 else "#D9731A" for v in sig["NES"]])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("NES (shPPP3CA vs shControl)")
ax.set_title(f"Hallmark gene sets (FDR < {FDR})")
plt.tight_layout()
plt.show()

## 1つの遺伝子セットを詳しく見る

GSEA の結果は、下のような図で表します。

- 上：遺伝子セットのスコアを、順位の上から順に足していった線。線が下に大きくへこむほど、そのセットの遺伝子が順位の下の方（shPPP3CA で減る側）に集まっています
- 中：そのセットの遺伝子が、順位のどこにあるかを示す縦線
- 下：順位を付けるのに使った値（`stat`）

`TERM` を書き換えると、ほかの遺伝子セットも見られます。

In [ ]:
TERM = "HALLMARK_E2F_TARGETS"

gsea.plot(terms=TERM)
plt.show()

## leading edge の遺伝子

偏りを生んでいる中心の遺伝子について、DESeq2 の結果を見てみます。

In [ ]:
lead = table.set_index("Term").loc[TERM, "Lead_genes"].split(";")
print(TERM, "の leading edge:", len(lead), "遺伝子")
res.loc[lead, ["baseMean", "log2FoldChange", "padj"]].head(15)

## E2F1 そのものと、E2F1 が働きを強める遺伝子

03 で見たとおり、*E2F1* の mRNA の量の変化は小さいものでした。一方、E2F が働きを強める遺伝子の集まりは、shPPP3CA でそろって減る側に偏っています。

In [ ]:
e2f1 = res.loc["E2F1"]
print(f"E2F1 の mRNA: log2FC = {e2f1['log2FoldChange']:.2f}, padj = {e2f1['padj']:.2g}")
e2f = table.set_index("Term").loc["HALLMARK_E2F_TARGETS"]
print(f"HALLMARK_E2F_TARGETS: NES = {e2f['NES']:.2f}, FDR q-val = {e2f['FDR q-val']:.2g}")

論文は、カルシニューリンが E2F1 のタンパク質を壊れにくくする、と報告しています。この考えに立つと、*PPP3CA* を減らすと E2F1 のタンパク質が減り、その結果 E2F1 が働きを強める遺伝子の RNA が減る、と説明できます。

RNA-seq で測れるのは RNA の量で、タンパク質の量は測っていません。この結果は論文の考えと合いますが、それだけで E2F1 のタンパク質が減ったことを示すものではありません。確かめるには、ウェスタンブロットなどでタンパク質を測る実験が必要です。

## 確かめよう

1. shPPP3CA で最も強く減る側に偏った遺伝子セットは何ですか。2番目と3番目は何ですか。それぞれどんな過程に関わりますか。
2. 増える側に偏った遺伝子セットには、どんなものがありますか。
3. *E2F1* の mRNA の量の変化は小さいのに、E2F が働きを強める遺伝子の集まりが減る側に偏るのは、なぜだと考えられますか。

In [ ]:
table.sort_values("NES").to_csv("gsea_results.csv", index=False)

## 結果をダウンロードする

下のセルを実行すると、ここまでの結果をまとめた `results_05.zip` がパソコンにダウンロードされます。

In [ ]:
import zipfile

OUT = "results_05.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ['counts_filtered.csv', 'vst.csv', 'deseq2_results.csv', 'normalized_counts.csv', 'significant_genes.csv', 'volcano.png', 'gsea_results.csv']:
        z.write(name)
print(OUT, "に入れたファイル:", zipfile.ZipFile(OUT).namelist())

try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass  # Colab 以外で実行したときは、手元のフォルダに保存される

## おわりに

この実習では、カウント表から出発して、次の順に解析しました。

1. カウント表の読み方（01）
2. 試料の品質と、試料どうしの似かた（02）
3. DESeq2 による2つの条件の比較（03）
4. ボルケーノプロットなどの図（04）
5. 遺伝子セットごとの解析（05）

最後にダウンロードした `results_05.zip` に、各段階の結果がまとめて入っています。

### 出典

- データ：Sato Y, et al. Calcineurin-mediated dephosphorylation stabilizes E2F1 protein by suppressing binding of the FBXW7 ubiquitin ligase subunit. *PNAS* 121, e2414618121 (2024). DDBJ DRA011729
- 遺伝子セット：MSigDB Hallmark gene sets v2024.1.Hs（Liberzon A, et al. *Cell Systems* 1, 417–425 (2015)）。CC BY 4.0
- 解析の道具：PyDESeq2（Muzellec B, et al. *Bioinformatics* 39, btad547 (2023)）、GSEApy（Fang Z, et al. *Bioinformatics* 39, btac757 (2023)）